# 06. Dielectric Model — k 데이터 수집 및 조인

Materials Project API에서 유전율(dielectric constant) 데이터를 수집하고,
기존 high-k 후보 소재(featurized_highk.csv)와 조인하여 모델 학습 데이터셋을 구축합니다.

**목표**: DRAM high-k 유전체 후보 소재의 유전율 예측 모델 개발

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
from mp_api.client import MPRester

load_dotenv('../.env')
API_KEY = os.getenv('MP_API_KEY')

with MPRester(API_KEY) as mpr:
    docs = mpr.materials.dielectric.search(
        fields=["material_id", "e_total", "e_electronic", "e_ionic"]
    )

# Handle both dict and object responses from mp-api
def get_attr(d, key):
    if isinstance(d, dict):
        return d.get(key)
    return getattr(d, key, None)

dielectric_df = pd.DataFrame([{
    'material_id': get_attr(d, 'material_id'),
    'e_total': float(get_attr(d, 'e_total')),
    'e_electronic': float(get_attr(d, 'e_electronic')) if get_attr(d, 'e_electronic') is not None else None,
    'e_ionic': float(get_attr(d, 'e_ionic')) if get_attr(d, 'e_ionic') is not None else None
} for d in docs if get_attr(d, 'e_total') is not None])

print(f"MP dielectric 데이터: {len(dielectric_df)}개")
print(f"e_total 범위: {dielectric_df['e_total'].min():.1f} ~ {dielectric_df['e_total'].max():.1f}")
print(f"중앙값: {dielectric_df['e_total'].median():.1f}")
dielectric_df.head()

MP dielectric 데이터: 7332개
e_total 범위: 1.2 ~ 126575.3
중앙값: 11.7


,material_id,e_total,e_electronic,e_ionic
0,mp-14,10.555159,9.692769,0.862390
1,mp-19,58.937522,43.624508,15.313013
2,mp-32,25.853525,25.853525,0.000000
3,mp-66,5.829714,5.829714,0.000000
4,mp-77,2.953744,2.832110,0.121634


In [2]:
featurized = pd.read_csv('../data/featurized_highk.csv')
print(f"featurized_highk: {len(featurized)}개")

merged = featurized.merge(dielectric_df, on='material_id', how='inner')
print(f"조인 후 (k 보유 후보): {len(merged)}개")
print(f"전체 {len(featurized)}개 중 {len(merged)/len(featurized)*100:.1f}% 가 k 데이터 보유")

# e_total 분포 확인
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(merged['e_total'], bins=50, color='steelblue', edgecolor='white')
plt.xlabel('e_total (dielectric constant)')
plt.ylabel('Count')
plt.title('Raw e_total Distribution')

plt.subplot(1, 2, 2)
plt.hist(np.log10(merged['e_total'].clip(lower=0.1)), bins=50, color='darkorange', edgecolor='white')
plt.xlabel('log10(e_total)')
plt.ylabel('Count')
plt.title('log10 Distribution')
plt.tight_layout()
plt.savefig('../data/dielectric_distribution.png', dpi=150)
plt.show()

merged.to_csv('../data/highk_with_dielectric.csv', index=False)
print("저장 완료: data/highk_with_dielectric.csv")

featurized_highk: 3914개
조인 후 (k 보유 후보): 624개
전체 3914개 중 15.9% 가 k 데이터 보유
Plot saved: data/dielectric_distribution.png
저장 완료: data/highk_with_dielectric.csv
